In [ ]:
!pip install scikit-surprise


In [ ]:
import pandas as pd
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split


In [ ]:
user_item_matrix = pd.read_csv("user_item_matrix (3).csv", index_col=0)
user_item_matrix.head()


,45,260,536,881,1008,1154,1194,1313,1360,1364,...,48377,48455,48679,48720,48825,49044,49141,49236,49273,49416
user_id,,,,,,,,,,,,,,,,,,,,,
140,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
209,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
273,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
473,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
545,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
data_long = user_item_matrix.stack().reset_index()
data_long.columns = ['user_id', 'product_id', 'interaction']

# Remove zero interactions
data_long = data_long[data_long['interaction'] > 0]


In [7]:
reader = Reader(rating_scale=(1, 3))

data = Dataset.load_from_df(
    data_long[['user_id', 'product_id', 'interaction']],
    reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)


In [8]:
model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02
)

model.fit(trainset)


In [9]:
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print("RMSE:", rmse)


RMSE: 0.0069
RMSE: 0.006928995152473663


In [10]:
models = {
    "SVD_20_epochs": SVD(n_epochs=20, n_factors=50),
    "SVD_30_epochs": SVD(n_epochs=30, n_factors=50)
}

for name, m in models.items():
    m.fit(trainset)
    preds = m.test(testset)
    rmse = accuracy.rmse(preds, verbose=False)
    print(f"{name} RMSE: {rmse}")


SVD_20_epochs RMSE: 0.015393446572815122
SVD_30_epochs RMSE: 0.009771625856588292


In [11]:
def recommend_products(user_id, model, user_item_matrix, top_n=5):
    user_items = user_item_matrix.loc[user_id]
    unseen_items = user_items[user_items == 0].index

    predictions = [(item, model.predict(user_id, item).est) for item in unseen_items]
    predictions.sort(key=lambda x: x[1], reverse=True)

    return predictions[:top_n]


In [12]:
recommend_products(user_item_matrix.index[0], model, user_item_matrix)


[('21938', 1.2118320609464568),
 ('9020', 1.1632312043981188),
 ('41117', 1.1569634560204811),
 ('5994', 1.150731217019005),
 ('13870', 1.1478754171228394)]